# Phase 7 — the working configuration on all five folds, and an ensemble

Phase 6b adopted HRNet-W32 at 1024 (run *e*) on fold 0 alone: PQ 0.3815. One
fold is 142 frames, and the same configuration has moved by 0.005 between two
runs, so whether the gain is real was left open. This notebook trains the same
configuration on all five folds, fold 0 included, and answers three questions.

1. **Does it hold up across the data?** Each fold's model is scored on the
   frames it did not see. The five scores are pooled — matches and misses
   added up, as if all 707 frames had been scored at once — which is the
   cross-validated estimate for this configuration.
2. **Does averaging the five models help on the test set?** An ensemble has no
   held-out frames here, since together the five models have seen all of
   them, so its only measurement is the leaderboard.
3. **How much of 0.3815 was the run?** Fold 0 is trained again with the same
   configuration and seed. GPU training is not bit-for-bit repeatable, so the
   distance between the new fold 0 and run *e* is a direct measure of what one
   rerun moves.

Everything uses `configs/phase6/e_unet_hrnet.yaml` unchanged except the fold,
and the post-processing Phase 3 settled on: threshold 0.5, minimum area 400,
rejoining within 24 pixels.

**Before running**, in the notebook settings:

1. Accelerator: **GPU T4 x2**
2. Internet: **on**
3. Inputs: the competition data
4. **Save Version** with *Save output* on

The five folds train two at a time, one per card. Expected wall clock: five to
five and a half hours.

**Output.** `submission_fold1.csv` to `submission_fold4.csv`, one per fold's
model, and `submission_ensemble.csv` from the mean of all five models'
probabilities; `submission_fold0.csv` is written too; it comes from the retrained fold 0, not
from run *e*, so it need not match what was already submitted. Kaggle saves at most 500 output files, so no
per-frame maps are written.

**Re-running after a failure.** Attach this notebook's earlier version as an
input as well. Whatever it finished is copied back and skipped.

## 1. Clone the repository and put it on the path

Cloned into `/tmp`, not `/kaggle/working`: the clone is over a hundred files,
and the saved output has room for 500.

In [ ]:
import sys

REPO_URL = "https://github.com/KeiichiIto1978/Solar_Filament_Segmentation_Challenge_2026.git"
REF = "feat/phase7-cv-ensemble"  # branch or commit hash
CHECKOUT = "/tmp/repo"

!rm -rf {CHECKOUT}
!git clone -q --branch {REF} {REPO_URL} {CHECKOUT}
!cd {CHECKOUT} && git log --oneline -1

# Only what Kaggle does not already have; torch stays as it is.
!pip install -q "segmentation-models-pytorch>=0.5"

if f"{CHECKOUT}/src" not in sys.path:
    sys.path.insert(0, f"{CHECKOUT}/src")

In [ ]:
import segmentation_models_pytorch as smp
import torch

import filament

print("filament", filament.__version__)
print("smp", smp.__version__)
print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.device_count())

## 2. Point the package at the competition data

In [ ]:
import os
from pathlib import Path

from filament.paths import load_paths

ANNOTATION_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"

candidates = sorted(Path("/kaggle/input").glob(f"**/{ANNOTATION_NAME}"))
if not candidates:
    raise SystemExit(
        "Could not find the annotation file under /kaggle/input. "
        "Attach the competition data to this notebook first."
    )

os.environ["MAGFILO_ROOT"] = str(candidates[0].parent.parent)
paths = load_paths().require_dataset()
print("MAGFILO_ROOT =", os.environ["MAGFILO_ROOT"])
print("train images:", len(list(paths.train_images.glob("*.jpeg"))))
print("test images: ", len(list(paths.test_images.glob("*.jpeg"))))

## 3. Five folds of one configuration

In [ ]:
import json
import shutil
from dataclasses import replace

from filament.data.coco import load_annotations
from filament.data.split import load_fold
from filament.postprocess.join import DEFAULT_MAX_OFFSET
from filament.training.config import TrainConfig

CONFIG_PATH = f"{CHECKOUT}/configs/phase6/e_unet_hrnet.yaml"
FOLDS = [0, 1, 2, 3, 4]
# Run e of Phase 6b on fold 0, for comparison with the retrained fold 0.
RUN_E_FOLD0_PQ = 0.3815

# The configuration Phase 3 settled on, at 1024, spelled out in full so that
# nothing depends on a default.
SCORING = {"threshold": 0.5, "min_area": 400, "join_gap": 24.0, "join_offset": DEFAULT_MAX_OFFSET}

base = TrainConfig.from_yaml(CONFIG_PATH)
configs = {
    fold: replace(base, fold=fold, num_workers=2, output_dir=Path(f"/kaggle/working/fold{fold}"))
    for fold in FOLDS
}
dataset = load_annotations(paths.train_annotations)
val_stems = {fold: load_fold(fold, f"{CHECKOUT}/configs/splits").val for fold in FOLDS}
for fold in FOLDS:
    print(f"fold {fold}: {len(val_stems[fold])} validation frames")
print(f"{base.architecture} / {base.encoder}, {base.image_size}px, batch {base.batch_size}")
print(f"{base.epochs} epochs")

In [ ]:
# Kaggle empties /kaggle/working at the start of a session, so anything
# finished earlier is reachable only through the inputs.
RUN_FILES = ("best.pt", "config.yaml", "history.json", "train.log")

for fold in FOLDS:
    destination = configs[fold].output_dir
    destination.mkdir(parents=True, exist_ok=True)
    if (destination / "best.pt").exists():
        continue
    # Only this notebook's own earlier versions: run e is deliberately not
    # reused, because retraining fold 0 is one of the questions.
    sources = [item for item in Path("/kaggle/input").glob(f"**/fold{fold}") if item.is_dir()]
    for source in sources:
        if (source / "best.pt").exists():
            for name in (*RUN_FILES, f"eval_fold{fold}.json"):
                if (source / name).exists():
                    shutil.copy(source / name, destination / name)
            print(f"fold {fold}: restored from {source}")
            break

## 4. Training, two folds at a time

Each fold is handed to `scripts/train.py` in its own process, pinned to one
card with `CUDA_VISIBLE_DEVICES`, so that each seeds its own random state as
run *e* did. With one card the folds run in turn; with none, nothing is
trained.

In [ ]:
import subprocess
import threading
import time

gpu_count = torch.cuda.device_count()
pending = [fold for fold in FOLDS if not (configs[fold].output_dir / "best.pt").exists()]
if gpu_count >= 2:
    QUEUES = {0: pending[0::2], 1: pending[1::2]}
elif gpu_count == 1:
    QUEUES = {0: pending}
else:
    QUEUES = {}
print(f"{gpu_count} GPU(s):", QUEUES)

print_lock = threading.Lock()
failures = {}


def run_queue(gpu, folds):
    for fold in folds:
        config = configs[fold]
        command = [
            sys.executable,
            f"{CHECKOUT}/scripts/train.py",
            "--config",
            CONFIG_PATH,
            "--fold",
            str(fold),
            "--output-dir",
            str(config.output_dir),
            "--num-workers",
            str(config.num_workers),
        ]
        environment = os.environ | {
            "CUDA_VISIBLE_DEVICES": str(gpu),
            "PYTHONPATH": f"{CHECKOUT}/src",
        }
        with print_lock:
            print(f"[gpu{gpu}] fold {fold}: started")
        started = time.perf_counter()
        # The log is kept beside the checkpoint, so it survives in the output.
        with open(config.output_dir / "train.log", "w") as log:
            process = subprocess.Popen(
                command,
                cwd=CHECKOUT,
                env=environment,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
            )
            for line in process.stdout:
                log.write(line)
                log.flush()
                if "Epoch" in line or "Error" in line or "Traceback" in line:
                    with print_lock:
                        print(f"[gpu{gpu}] fold {fold}: {line.rstrip()}")
            exit_code = process.wait()
        with print_lock:
            minutes = (time.perf_counter() - started) / 60
            print(f"[gpu{gpu}] fold {fold}: exit {exit_code} after {minutes:.1f} min")
        if exit_code != 0:
            failures[fold] = exit_code


threads = [threading.Thread(target=run_queue, args=item, daemon=True) for item in QUEUES.items()]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

for fold, exit_code in failures.items():
    log_path = configs[fold].output_dir / "train.log"
    print(f"fold {fold} FAILED with exit code {exit_code}; last lines of {log_path}:")
    print("".join(log_path.read_text().splitlines(keepends=True)[-20:]))

## 5. Each fold on the frames it did not see

The adoption rule and the phases before this one were read on fold 0. Here
every fold gets the same reading. The pooled line adds up matches and misses
across the five, which weights each fold by how many filaments it holds; the
mean of the five PQs is shown beside it for comparison.

In [ ]:
import pandas as pd

from filament.evaluation import evaluate
from filament.metrics.pq import PQResult, pool_pq
from filament.submit.rle import masks_to_gt_df
from filament.training.loop import HISTORY_NAME, load_checkpoint

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

scores = {}
for fold in FOLDS:
    config = configs[fold]
    eval_path = config.output_dir / f"eval_fold{fold}.json"
    if eval_path.exists():
        scores[fold] = json.loads(eval_path.read_text())
        print(f"fold {fold}: already scored, PQ {scores[fold]['pq']}")
        continue
    checkpoint = config.output_dir / "best.pt"
    if not checkpoint.exists():
        print(f"fold {fold}: no checkpoint, not scored")
        continue
    model, _ = load_checkpoint(checkpoint)
    evaluation, _ = evaluate(
        model,
        dataset,
        paths.train_images,
        val_stems[fold],
        size=config.image_size,
        device=DEVICE,
        gt_df=masks_to_gt_df(dataset, val_stems[fold]),
        **SCORING,
    )
    history = json.loads((config.output_dir / HISTORY_NAME).read_text())
    scores[fold] = evaluation.to_dict() | {
        "fold": fold,
        "best_epoch": min(history, key=lambda item: item["val_loss"])["epoch"],
        "best_val_loss": round(min(item["val_loss"] for item in history), 4),
        "training_minutes": round(sum(item["seconds"] for item in history) / 60, 1),
    }
    eval_path.write_text(json.dumps(scores[fold], indent=2))
    print(f"fold {fold}: {evaluation}")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

COLUMNS = ["pq", "sq", "rq", "tp", "fp", "fn", "fused", "split", "best_epoch", "training_minutes"]
table = pd.DataFrame(
    {f"fold {fold}": {key: row.get(key) for key in COLUMNS} for fold, row in scores.items()}
).T
table = table.apply(pd.to_numeric, errors="coerce")
pooled = pool_pq(
    PQResult(pq=row["pq"], sq=row["sq"], rq=row["rq"], tp=row["tp"], fp=row["fp"], fn=row["fn"])
    for row in scores.values()
)
table.loc["pooled"] = {
    "pq": round(pooled.pq, 4),
    "sq": round(pooled.sq, 4),
    "rq": round(pooled.rq, 4),
    "tp": pooled.tp,
    "fp": pooled.fp,
    "fn": pooled.fn,
}
table.loc["mean of folds"] = (
    table.loc[[f"fold {fold}" for fold in scores], ["pq", "sq", "rq"]].mean().round(4)
)
if 0 in scores:
    print(f"fold 0 retrained: PQ {scores[0]['pq']} against run e's {RUN_E_FOLD0_PQ}")
Path("/kaggle/working/cv_summary.json").write_text(
    json.dumps(
        {
            "folds": scores,
            "pooled": {
                "pq": pooled.pq,
                "sq": pooled.sq,
                "rq": pooled.rq,
                "tp": pooled.tp,
                "fp": pooled.fp,
                "fn": pooled.fn,
            },
        },
        indent=2,
    )
)
table

## 6. The submissions

One pass over the test frames with every fold's model loaded. Each frame's
probability map is computed once per model; the single-fold submissions are
drawn from those maps, and the ensemble from their mean, all through the same
post-processing as the scores above.

The overlap check runs on every file before it is left in the output. Kaggle
rejects a submission whose masks share a pixel, and the rejected attempt still
counts against the five allowed per day.

In [ ]:
from filament.data.disk import detect_disk
from filament.data.image import load_grayscale
from filament.evaluation import predict_probability
from filament.metrics.overlap import check_no_overlap
from filament.postprocess.instances import extract_instances, instances_to_rows
from filament.submit.rle import write_submission

models = {}
for fold in FOLDS:
    checkpoint = configs[fold].output_dir / "best.pt"
    if checkpoint.exists():
        models[fold], _ = load_checkpoint(checkpoint)
print("models loaded for folds", sorted(models))
if not models:
    raise SystemExit("No fold has a model, so there is nothing to submit.")

size = base.image_size
test_stems = sorted(path.stem for path in paths.test_images.glob("*.jpeg"))
rows = {f"fold{fold}": [] for fold in models}
rows["ensemble"] = []
for position, stem in enumerate(test_stems, start=1):
    frame = load_grayscale(paths.test_images / f"{stem}.jpeg")
    disk = detect_disk(frame).scaled(size / frame.shape[0])
    maps = {fold: predict_probability(model, frame, size, DEVICE) for fold, model in models.items()}
    for fold, probability in maps.items():
        rows[f"fold{fold}"].extend(
            instances_to_rows(stem, extract_instances(probability, disk=disk, **SCORING))
        )
    mean = sum(maps.values()) / len(maps)
    rows["ensemble"].extend(instances_to_rows(stem, extract_instances(mean, disk=disk, **SCORING)))
    if position % 30 == 0:
        print(f"{position}/{len(test_stems)}")

written = {}
for name, entries in rows.items():
    if name == "ensemble" and len(models) < 2:
        print("ensemble: skipped, fewer than two models")
        continue
    path = Path(f"/kaggle/working/submission_{name}.csv")
    write_submission(entries, path)
    # Raises before the file is counted as written if any two masks touch.
    check_no_overlap(path)
    frames = len({entry[0].rsplit("_", 1)[0] for entry in entries})
    written[name] = {"masks": len(entries), "frames_with_predictions": frames}
    print(f"{path.name}: {len(entries)} masks over {frames} of {len(test_stems)} frames")

Path("/kaggle/working/submission_info.json").write_text(
    json.dumps(
        {"commit": REF, "folds": sorted(models), "scoring": SCORING, "files": written}, indent=2
    )
)

In [ ]:
# Kaggle keeps at most 500 output files and drops the rest without an error.
saved = [item for item in Path("/kaggle/working").rglob("*") if item.is_file()]
print(f"{len(saved)} files in /kaggle/working")
if len(saved) > 500:
    raise SystemExit(f"{len(saved)} output files; Kaggle would silently drop some of them.")

## 7. What to record

Into the lab notebook:

- each fold's PQ, SQ, RQ, TP, FP, FN, fused and split, best epoch and training
  time; the pooled line and the mean of folds
- the retrained fold 0 against run *e*'s 0.3815: what one rerun moves
- how far fold 0 sits from the other four: it is the fold every earlier
  decision was read on
- the leaderboard score of each submission next to its fold's PQ, and the
  ensemble's; the commit hash this notebook cloned

The leaderboard shows two decimals and the public part is about half the test
set, so differences below 0.01 there are not readable.